# Physics-Informed Neural Networks for the Time-Dependent Schrödinger Equation

## Mathematical Framework

The **Time-Dependent Schrödinger Equation (TDSE)** in one spatial dimension (with $\hbar = m = 1$):

$$i\frac{\partial\psi}{\partial t} = \hat{H}\psi = \left[-\frac{1}{2}\frac{\partial^2}{\partial x^2} + V(x)\right]\psi(x,t)$$

Since $\psi \in \mathbb{C}$, we split into real and imaginary parts $\psi = \psi_r + i\psi_i$ yielding two **coupled real PDEs**:

$$\frac{\partial\psi_r}{\partial t} = -\frac{1}{2}\frac{\partial^2\psi_i}{\partial x^2} + V(x)\psi_i$$

$$\frac{\partial\psi_i}{\partial t} = +\frac{1}{2}\frac{\partial^2\psi_r}{\partial x^2} - V(x)\psi_r$$

### Madelung Representation

Writing $\psi = \sqrt{\rho}\,e^{i\theta}$ where $\rho=|\psi|^2$ is the probability density and $\theta = \arctan(\psi_i/\psi_r)$ is the phase, the TDSE becomes the **Madelung equations** (quantum hydrodynamics):

$$\frac{\partial\rho}{\partial t} + \frac{\partial}{\partial x}\left(\rho\frac{\partial\theta}{\partial x}\right) = 0 \quad \text{(continuity)}$$

$$\frac{\partial\theta}{\partial t} + \frac{1}{2}\left(\frac{\partial\theta}{\partial x}\right)^2 + V - \frac{1}{2\sqrt{\rho}}\frac{\partial^2\sqrt{\rho}}{\partial x^2} = 0 \quad \text{(quantum Euler)}$$

The last term $Q = -\frac{1}{2\sqrt\rho}\frac{\partial^2\sqrt\rho}{\partial x^2}$ is the **Bohm quantum potential**.

### Probability Current

The **probability current density** describes the flow of probability:

$$J(x,t) = \frac{\hbar}{m}\,\text{Im}\!\left(\psi^*(x,t)\frac{\partial\psi}{\partial x}\right) = \psi_r\frac{\partial\psi_i}{\partial x} - \psi_i\frac{\partial\psi_r}{\partial x}$$

satisfying the continuity equation $\frac{\partial\rho}{\partial t} + \frac{\partial J}{\partial x} = 0$.

### Ehrenfest Theorem

Quantum expectation values obey classical equations of motion:

$$\frac{d\langle x\rangle}{dt} = \langle p\rangle = \int x\,J(x,t)\,dx\bigg/\int |x|\,dx \approx \int x\,J(x,t)\,dx$$

$$\frac{d\langle p\rangle}{dt} = -\left\langle \frac{\partial V}{\partial x}\right\rangle = -\int \rho(x,t)\frac{\partial V}{\partial x}\,dx$$

### PINN Formulation

We train two networks $\hat\psi_r(x,t;\theta_r)$ and $\hat\psi_i(x,t;\theta_i)$ (or a single two-headed network) to minimize:

$$\mathcal{L} = \underbrace{\mathcal{L}_{\text{PDE},r} + \mathcal{L}_{\text{PDE},i}}_{\text{TDSE residuals}} + \lambda_{\text{ic}}\underbrace{\mathcal{L}_{\text{IC}}}_{\text{initial condition}} + \lambda_{\text{bc}}\underbrace{\mathcal{L}_{\text{BC}}}_{\text{boundary conditions}} + \lambda_{\text{norm}}\underbrace{\mathcal{L}_{\text{norm}}}_{\|\psi\|^2=1}$$

$$\mathcal{L}_{\text{PDE},r} = \frac{1}{N_c}\sum_{i}\left|\frac{\partial\hat\psi_r}{\partial t} + \frac{1}{2}\frac{\partial^2\hat\psi_i}{\partial x^2} - V\hat\psi_i\right|^2$$

$$\mathcal{L}_{\text{PDE},i} = \frac{1}{N_c}\sum_{i}\left|\frac{\partial\hat\psi_i}{\partial t} - \frac{1}{2}\frac{\partial^2\hat\psi_r}{\partial x^2} + V\hat\psi_r\right|^2$$

### Benchmark Initial Condition: Gaussian Wavepacket

A coherent state (minimum-uncertainty Gaussian wavepacket) centered at $x_0$ with initial momentum $k_0$:

$$\psi(x, 0) = \frac{1}{(\pi\sigma^2)^{1/4}}\exp\!\left(-\frac{(x-x_0)^2}{2\sigma^2} + ik_0 x\right)$$

The exact time-evolution in a free-particle potential ($V=0$) is:

$$\psi(x,t) = \frac{1}{(π)^{1/4}\sqrt{\sigma + it/\sigma}}\exp\!\left[-\frac{(x-x_0-k_0 t)^2}{2\sigma(\sigma+it/\sigma)} + ik_0 x - \frac{ik_0^2 t}{2}\right]$$

Key observables: center travels classically $\langle x\rangle(t) = x_0 + k_0 t$; width spreads as $\sigma(t) = \sqrt{\sigma^2 + t^2/\sigma^2}$.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
import os

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',   'grid.color': '#21262d',
    'text.color': '#e6edf3',       'axes.labelcolor': '#8b949e',
    'xtick.color': '#8b949e',      'ytick.color': '#8b949e',
    'font.size': 11, 'axes.titlesize': 13,
})

torch.manual_seed(42)
np.random.seed(42)
os.makedirs('../outputs', exist_ok=True)

# ─── Exact Gaussian wavepacket solution (free particle, V=0) ────────────────
def exact_wavepacket(x, t, x0=0.0, k0=2.0, sigma=0.5):
    """Exact free-particle Gaussian wavepacket. Returns (psi_r, psi_i)."""
    x, t = np.asarray(x), np.asarray(t)
    sigma_t = sigma + 1j * t / sigma         # complex width
    norm    = (np.pi)**0.25 * np.sqrt(sigma_t)
    exponent = -(x - x0 - k0*t)**2 / (2 * sigma * sigma_t) + 1j * k0 * x - 0.5j * k0**2 * t
    psi     = np.exp(exponent) / norm
    return np.real(psi), np.imag(psi)

# Plot the exact wavepacket at several times
t_snaps = [0.0, 0.25, 0.5, 0.75, 1.0]
x_arr   = np.linspace(-5, 8, 500)
x0, k0, sigma = 0.0, 2.0, 0.5

fig, axes = plt.subplots(2, len(t_snaps), figsize=(18, 7), facecolor='#0d1117')
fig.suptitle('Exact Gaussian Wavepacket $\\psi(x,t)$ — Free Particle ($V=0$)', color='#e6edf3', fontsize=14)

cmap_psi = cm.get_cmap('plasma')
for j, t_j in enumerate(t_snaps):
    psi_r, psi_i = exact_wavepacket(x_arr, t_j, x0, k0, sigma)
    rho  = psi_r**2 + psi_i**2
    # Centre and width tracking
    xc   = x0 + k0 * t_j
    sig_t = np.sqrt(sigma**2 + t_j**2 / sigma**2)

    ax_top = axes[0, j]
    ax_top.set_facecolor('#161b22')
    for sp in ax_top.spines.values(): sp.set_edgecolor('#30363d')
    ax_top.plot(x_arr, psi_r, '-', color='#58a6ff', lw=2,   label=r'Re[$\psi$]')
    ax_top.plot(x_arr, psi_i, '--', color='#ffa657', lw=2,  label=r'Im[$\psi$]')
    ax_top.axvline(xc, color='#ff7b72', lw=0.8, ls=':')
    ax_top.set_title(f'$t={t_j:.2f}$  $⟨x⟩={xc:.2f}$  $σ={sig_t:.3f}$', color='#e6edf3', fontsize=11)
    ax_top.set_xlabel('$x$', color='#8b949e')
    ax_top.set_xlim(-5, 8)
    if j == 0:
        ax_top.set_ylabel(r'$\psi(x,t)$', color='#8b949e')
        ax_top.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=9)

    ax_bot = axes[1, j]
    ax_bot.set_facecolor('#161b22')
    for sp in ax_bot.spines.values(): sp.set_edgecolor('#30363d')
    ax_bot.fill_between(x_arr, rho, alpha=0.5, color='#3fb950')
    ax_bot.plot(x_arr, rho, '-', color='#3fb950', lw=2)
    ax_bot.axvline(xc, color='#ff7b72', lw=0.8, ls=':')
    ax_bot.set_xlabel('$x$', color='#8b949e')
    ax_bot.set_xlim(-5, 8)
    if j == 0:
        ax_bot.set_ylabel(r'$|\psi|^2$', color='#8b949e')

plt.tight_layout()
plt.savefig('../outputs/schrodinger_exact_snapshots.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved → outputs/schrodinger_exact_snapshots.png')
print(f'Wavepacket parameters: x0={x0}, k0={k0}, σ₀={sigma}')
print(f'Width at t=1.0: σ(1)={np.sqrt(sigma**2 + 1/sigma**2):.4f}  (σ₀={sigma})')
print(f'Norm at t=0: {np.trapz(exact_wavepacket(x_arr,0)[0]**2 + exact_wavepacket(x_arr,0)[1]**2, x_arr):.6f}')

In [ ]:
# ── ComplexPINN Architecture: Dual-Output Real/Imaginary Network ──────────
class ComplexPINN(nn.Module):
    """
    Two-headed network for TDSE: shared trunk, separate heads for ψ_r and ψ_i.
    Input:  (x, t)  [2D]
    Output: (ψ_r, ψ_i) [2D]
    Hard boundary conditions via tanh-based envelope:
        ψ(±L, t) ≈ 0  ⟹  multiply by (L² - x²) / L²
    """
    def __init__(self, hidden=64, n_layers=5, x_domain=(-5.0, 8.0)):
        super().__init__()
        self.x_min, self.x_max = x_domain
        act = nn.Tanh()
        # Shared trunk
        trunk = [nn.Linear(2, hidden), act]
        for _ in range(n_layers - 2):
            trunk += [nn.Linear(hidden, hidden), act]
        self.trunk = nn.Sequential(*trunk)
        # Two independent heads
        self.head_r = nn.Linear(hidden, 1)
        self.head_i = nn.Linear(hidden, 1)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def bc_envelope(self, x):
        """Smooth window function that vanishes at domain boundaries."""
        L = (self.x_max - self.x_min) / 2.0
        x_c = (x - (self.x_max + self.x_min) / 2.0) / L       # map to [-1, 1]
        return (1.0 - x_c**2).clamp(min=0.0)                    # > 0 in interior, 0 at walls

    def forward(self, x, t):
        xt    = torch.cat([x, t], dim=-1)
        feat  = self.trunk(xt)
        psi_r = self.head_r(feat) * self.bc_envelope(x)
        psi_i = self.head_i(feat) * self.bc_envelope(x)
        return psi_r, psi_i

    def probability_density(self, x, t):
        psi_r, psi_i = self.forward(x, t)
        return psi_r**2 + psi_i**2

# ── Domain Setup ────────────────────────────────────────────────────────
X_MIN, X_MAX = -5.0, 8.0
T_MIN, T_MAX = 0.0, 1.0
x0, k0, sigma = 0.0, 2.0, 0.5

N_interior = 4000      # PDE collocation points
N_ic       = 500       # initial condition τ=0 points
N_bc       = 200       # boundary condition points

torch.manual_seed(42)

# Interior: random (x, t) in domain
x_int = torch.rand(N_interior, 1) * (X_MAX - X_MIN) + X_MIN
t_int = torch.rand(N_interior, 1) * (T_MAX - T_MIN) + T_MIN

# Initial condition: t=0, random x
x_ic = torch.rand(N_ic, 1) * (X_MAX - X_MIN) + X_MIN
t_ic = torch.zeros(N_ic, 1)
psi_r_ic_exact, psi_i_ic_exact = exact_wavepacket(x_ic.numpy(), 0.0, x0, k0, sigma)
psi_r0 = torch.tensor(psi_r_ic_exact, dtype=torch.float32)
psi_i0 = torch.tensor(psi_i_ic_exact, dtype=torch.float32)

# Evaluating norm at t=0 for reference
x_eval  = torch.linspace(X_MIN, X_MAX, 512).unsqueeze(1)
pr, pi  = exact_wavepacket(x_eval.numpy().flatten(), 0.0, x0, k0, sigma)
rho_ic  = pr**2 + pi**2
dx_e    = (X_MAX - X_MIN) / 511
norm_ic = np.trapz(rho_ic, x_eval.numpy().flatten())
print(f'Initial norm (exact) ∫|ψ|²dx = {norm_ic:.5f}  (should be ≈1)')

# Instantiate model
model   = ComplexPINN(hidden=64, n_layers=5, x_domain=(X_MIN, X_MAX))
n_params = sum(p.numel() for p in model.parameters())
print(f'ComplexPINN: {n_params:,} parameters')
print(f'Domain: x ∈ [{X_MIN}, {X_MAX}], t ∈ [{T_MIN}, {T_MAX}]')
print(f'Collocation: {N_interior} interior, {N_ic} IC, {N_bc} BC points')

In [ ]:
# ── TDSE Residual Computation ─────────────────────────────────────────────
def tdse_residual(model, x, t, V_fn=None):
    """
    Compute TDSE residuals for real and imaginary parts.

    TDSE: i ∂ψ/∂t = -½ ∂²ψ/∂x² + V(x)ψ
    Splitting into real/imag:
      ∂ψ_r/∂t = -½ ∂²ψ_i/∂x² + V ψ_i
      ∂ψ_i/∂t = +½ ∂²ψ_r/∂x² - V ψ_r

    Returns: res_r, res_i  (should both be zero at a valid solution)
    """
    x = x.requires_grad_(True)
    t = t.requires_grad_(True)

    psi_r, psi_i = model(x, t)

    # Time derivatives
    dpsi_r_dt, = torch.autograd.grad(psi_r.sum(), t, create_graph=True)
    dpsi_i_dt, = torch.autograd.grad(psi_i.sum(), t, create_graph=True)

    # First spatial derivatives
    dpsi_r_dx,  = torch.autograd.grad(psi_r.sum(), x, create_graph=True)
    dpsi_i_dx,  = torch.autograd.grad(psi_i.sum(), x, create_graph=True)

    # Second spatial derivatives
    d2psi_r_dx2, = torch.autograd.grad(dpsi_r_dx.sum(), x, create_graph=True)
    d2psi_i_dx2, = torch.autograd.grad(dpsi_i_dx.sum(), x, create_graph=True)

    V = V_fn(x) if V_fn else torch.zeros_like(x)

    res_r = dpsi_r_dt - (-0.5 * d2psi_i_dx2 + V * psi_i)     # ∂ψ_r/∂t − rhs
    res_i = dpsi_i_dt - ( 0.5 * d2psi_r_dx2 - V * psi_r)     # ∂ψ_i/∂t − rhs

    return res_r, res_i, dpsi_r_dx, dpsi_i_dx

# ── Loss function ────────────────────────────────────────────────────────
def compute_loss(model, x_int, t_int, x_ic, t_ic, psi_r0, psi_i0,
                 lam_ic=10.0, lam_norm=2.0, V_fn=None):
    """Composite TDSE loss."""
    # PDE residuals
    res_r, res_i, _, _ = tdse_residual(model, x_int, t_int, V_fn)
    L_pde = (res_r**2 + res_i**2).mean()

    # Initial condition loss
    pr_pred, pi_pred = model(x_ic, t_ic)
    L_ic = ((pr_pred - psi_r0)**2 + (pi_pred - psi_i0)**2).mean()

    # Norm conservation: ∫|ψ(x, t)|² dx ≈ 1 at sampled times
    x_line = torch.linspace(X_MIN, X_MAX, 256).unsqueeze(1)
    t_rand = torch.rand(8, 1) * T_MAX
    L_norm = torch.zeros(1)
    dx_nn  = (X_MAX - X_MIN) / 255.0
    for t_samp in t_rand:
        t_rep = t_samp.expand(256, 1)
        pr, pi = model(x_line, t_rep)
        norm_v = (pr**2 + pi**2).sum() * dx_nn
        L_norm = L_norm + (norm_v - 1.0)**2
    L_norm = L_norm / 8.0

    total = L_pde + lam_ic * L_ic + lam_norm * L_norm
    return total, L_pde, L_ic, L_norm

# ── Training Loop ─────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[2000, 4000, 6000], gamma=0.4)

N_EPOCHS  = 6000
PRINT_EVERY = 500
history = {'epoch': [], 'total': [], 'pde': [], 'ic': [], 'norm': []}

print(f'Training TDSE-PINN for {N_EPOCHS} epochs on free-particle Gaussian wavepacket ...')
print(f'{"Epoch":>7}  {"Total":>10}  {"PDE":>10}  {"IC":>10}  {"Norm":>10}  {"LR":>9}')
print('─' * 65)

for ep in range(1, N_EPOCHS + 1):
    optimizer.zero_grad()
    x_b = x_int.requires_grad_(False).detach().requires_grad_(True)
    t_b = t_int.requires_grad_(False).detach().requires_grad_(True)
    loss, L_pde, L_ic, L_norm = compute_loss(model, x_b, t_b,
                                              x_ic, t_ic, psi_r0, psi_i0)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    if ep % PRINT_EVERY == 0 or ep == 1:
        lr = optimizer.param_groups[0]['lr']
        print(f'{ep:>7}  {loss.item():>10.4e}  {L_pde.item():>10.4e}  '
              f'{L_ic.item():>10.4e}  {L_norm.item():>10.4e}  {lr:>9.2e}')
        history['epoch'].append(ep)
        history['total'].append(loss.item())
        history['pde'].append(L_pde.item())
        history['ic'].append(L_ic.item())
        history['norm'].append(L_norm.item())

# ── Loss convergence plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22')
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')

axes[0].semilogy(history['epoch'], history['pde'],   '-',  color='#58a6ff', lw=2, label='$\\mathcal{L}_{\\rm PDE}$')
axes[0].semilogy(history['epoch'], history['ic'],    '--', color='#3fb950', lw=2, label='$\\mathcal{L}_{\\rm IC}$')
axes[0].semilogy(history['epoch'], history['norm'],  ':',  color='#ffa657', lw=2, label='$\\mathcal{L}_{\\rm norm}$')
axes[0].semilogy(history['epoch'], history['total'], '-',  color='#d2a8ff', lw=2.5, alpha=0.6, label='Total')
axes[0].set_xlabel('Epoch', color='#8b949e'); axes[0].set_ylabel('Loss', color='#8b949e')
axes[0].set_title('TDSE-PINN Training Convergence', color='#e6edf3')
axes[0].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')
axes[0].tick_params(colors='#8b949e')

# Loss ratio: PDE vs IC
ratio = [p/max(i, 1e-20) for p, i in zip(history['pde'], history['ic'])]
axes[1].semilogy(history['epoch'], ratio, '-', color='#ff7b72', lw=2)
axes[1].axhline(1.0, color='#30363d', lw=1, ls='--')
axes[1].set_xlabel('Epoch', color='#8b949e'); axes[1].set_ylabel('$\\mathcal{L}_{\\rm PDE}/\\mathcal{L}_{\\rm IC}$', color='#8b949e')
axes[1].set_title('PDE/IC Loss Ratio (balance indicator)', color='#e6edf3')
axes[1].tick_params(colors='#8b949e')

plt.tight_layout()
plt.savefig('../outputs/schrodinger_convergence.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved → outputs/schrodinger_convergence.png')

In [ ]:
# ── Probability Density Spacetime Heatmap ────────────────────────────────
model.eval()
x_v = torch.linspace(X_MIN, X_MAX, 300).unsqueeze(1)
t_v_pts = np.linspace(T_MIN, T_MAX, 150)

rho_field    = np.zeros((150, 300))
rho_exact_f  = np.zeros((150, 300))

with torch.no_grad():
    for j, tj in enumerate(t_v_pts):
        t_rep = torch.full((300, 1), tj)
        pr, pi = model(x_v, t_rep)
        rho_field[j]   = (pr**2 + pi**2).numpy().flatten()
        pr_e, pi_e = exact_wavepacket(x_v.numpy().flatten(), tj, x0, k0, sigma)
        rho_exact_f[j] = pr_e**2 + pi_e**2

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor='#0d1117')
fig.suptitle('Probability Density $|\\psi(x,t)|^2$ — PINN vs Exact', color='#e6edf3', fontsize=14)
x_arr_v = x_v.numpy().flatten()

titles = ['Exact $|\\psi|^2$', 'PINN $|\\hat\\psi|^2$', 'Pointwise Error']
datas  = [rho_exact_f, rho_field, np.abs(rho_field - rho_exact_f)]
cmaps  = ['plasma', 'plasma', 'Reds']

for ax, data, ttl, cmap in zip(axes, datas, titles, cmaps):
    ax.set_facecolor('#161b22')
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
    im = ax.imshow(data, extent=[X_MIN, X_MAX, T_MAX, T_MIN],
                   aspect='auto', cmap=cmap, interpolation='bilinear')
    ax.set_xlabel('$x$', color='#8b949e'); ax.set_ylabel('$t$', color='#8b949e')
    ax.set_title(ttl, color='#e6edf3')
    ax.tick_params(colors='#8b949e')
    div = make_axes_locatable(ax)
    cax = div.append_axes('right', size='4%', pad=0.05)
    plt.colorbar(im, cax=cax)
    # Classical trajectory
    t_line = t_v_pts
    x_class= x0 + k0 * t_line
    mask = (x_class >= X_MIN) & (x_class <= X_MAX)
    ax.plot(x_class[mask], t_line[mask], 'w--', lw=1, alpha=0.5, label='Classical $⟨x⟩$')
    ax.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/schrodinger_density_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Max pointwise density error
max_err = np.max(np.abs(rho_field - rho_exact_f))
rms_err = np.sqrt(np.mean((rho_field - rho_exact_f)**2))
print(f'Max pointwise |ρ_pinn - ρ_exact| = {max_err:.5f}')
print(f'RMS pointwise error               = {rms_err:.5f}')
print('Saved → outputs/schrodinger_density_heatmap.png')

In [ ]:
# ── Probability Current J(x,t) and Continuity Equation ───────────────────
# J(x,t) = ψ_r ∂ψ_i/∂x  −  ψ_i ∂ψ_r/∂x
# Continuity: ∂ρ/∂t + ∂J/∂x = 0  ⟹  conservation of probability

def probability_current(model, x, t):
    """Compute J(x,t) = ψ_r ∂ψ_i/∂x − ψ_i ∂ψ_r/∂x via autograd."""
    x = x.requires_grad_(True)
    t = t.detach().requires_grad_(False)
    pr, pi = model(x, t)
    dpr_dx, = torch.autograd.grad(pr.sum(), x, create_graph=False, retain_graph=True)
    dpi_dx, = torch.autograd.grad(pi.sum(), x, create_graph=False)
    J = (pr * dpi_dx - pi * dpr_dx).detach()
    return J

t_snaps = [0.0, 0.25, 0.5, 0.75, 1.0]
x_s = torch.linspace(X_MIN, X_MAX, 400).unsqueeze(1)
dx_s = float((X_MAX - X_MIN) / 399)
x_np = x_s.numpy().flatten()

fig, axes = plt.subplots(2, len(t_snaps), figsize=(18, 8), facecolor='#0d1117')
fig.suptitle('PINN Solution vs Analytic: $|\\psi|^2$ and $J(x,t)$ Snapshots', color='#e6edf3', fontsize=14)

norm_vals_pinn  = []
norm_vals_exact = []
J_max_vals      = []
curr_residuals  = []   # ∂ρ/∂t + ∂J/∂x

for j, tj in enumerate(t_snaps):
    t_rep = torch.full((400, 1), tj)

    with torch.no_grad():
        pr, pi = model(x_s, t_rep)
    rho_p = (pr**2 + pi**2).numpy().flatten()

    pr_e, pi_e = exact_wavepacket(x_np, tj, x0, k0, sigma)
    rho_e = pr_e**2 + pi_e**2

    # Probability current
    J_pinn = probability_current(model, x_s, t_rep).numpy().flatten()
    J_exact = pr_e * np.gradient(pi_e, x_np) - pi_e * np.gradient(pr_e, x_np)

    norm_vals_pinn.append(np.trapz(rho_p, x_np))
    norm_vals_exact.append(np.trapz(rho_e, x_np))
    J_max_vals.append(np.max(np.abs(J_pinn)))

    # Density subplot
    ax_top = axes[0, j]
    ax_top.set_facecolor('#161b22')
    for sp in ax_top.spines.values(): sp.set_edgecolor('#30363d')
    ax_top.plot(x_np, rho_e, '-', color='#3fb950', lw=2.5,   label='Exact $|\\psi|^2$')
    ax_top.plot(x_np, rho_p, '--', color='#58a6ff', lw=1.8, label='PINN $|\\hat\\psi|^2$')
    ax_top.set_title(f'$t={tj:.2f}$', color='#e6edf3', fontsize=10)
    ax_top.set_xlabel('$x$', color='#8b949e')
    if j == 0:
        ax_top.set_ylabel('$|\\psi|^2$', color='#8b949e')
        ax_top.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=8)
    ax_top.tick_params(colors='#8b949e')

    # Current subplot
    ax_bot = axes[1, j]
    ax_bot.set_facecolor('#161b22')
    for sp in ax_bot.spines.values(): sp.set_edgecolor('#30363d')
    ax_bot.plot(x_np, J_exact, '-', color='#ffa657', lw=2.5, label='Exact $J$')
    ax_bot.plot(x_np, J_pinn,  '--', color='#d2a8ff', lw=1.8, label='PINN $J$')
    ax_bot.axhline(0, color='#30363d', lw=0.7)
    ax_bot.set_xlabel('$x$', color='#8b949e')
    if j == 0:
        ax_bot.set_ylabel('$J(x,t)$', color='#8b949e')
        ax_bot.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=8)
    ax_bot.tick_params(colors='#8b949e')

plt.tight_layout()
plt.savefig('../outputs/schrodinger_current_snapshots.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

print('Norm conservation  ∫|ψ|²dx:')
print(f'  {"t":>5s}  {"PINN":>10s}  {"Exact":>10s}  {"|Δ|":>10s}')
for j, tj in enumerate(t_snaps):
    print(f'  {tj:>5.2f}  {norm_vals_pinn[j]:>10.6f}  {norm_vals_exact[j]:>10.6f}  '
          f'{abs(norm_vals_pinn[j]-norm_vals_exact[j]):>10.6f}')
print('Saved → outputs/schrodinger_current_snapshots.png')

In [ ]:
# ── Ehrenfest Theorem: ⟨x⟩(t) and ⟨p⟩(t) Trajectories ──────────────────
# Classical analogue: d⟨x⟩/dt = ⟨p⟩  and  d⟨p⟩/dt = -⟨∂V/∂x⟩
# For V=0: ⟨x⟩(t) = x0 + k0*t  and  ⟨p⟩(t) = k0  (constant)
# For QHO (V = ½x²): ⟨x⟩(t) = x0 cos(t) + k0 sin(t)
#                   ⟨p⟩(t) = -x0 sin(t) + k0 cos(t)

t_fine  = np.linspace(T_MIN, T_MAX, 80)
x_grid  = x_s.numpy().flatten()
dx_g    = x_grid[1] - x_grid[0]

xmean_pinn  = []
pmean_pinn  = []
norm_time   = []

model.eval()
for tj in t_fine:
    t_rep = torch.full((400, 1), tj)
    x_s_req = x_s.requires_grad_(True)
    pr, pi = model(x_s_req, t_rep)
    rho = (pr**2 + pi**2).detach().numpy().flatten()

    # ⟨x⟩ = ∫ x |ψ|² dx
    nm  = np.trapz(rho, x_grid)
    xm  = np.trapz(x_grid * rho, x_grid) / max(nm, 1e-12)
    xmean_pinn.append(xm)
    norm_time.append(nm)

    # ⟨p⟩ = ∫ ψ* (-i ∂/∂x) ψ dx = ∫ (ψ_r ∂ψ_i/∂x − ψ_i ∂ψ_r/∂x) dx
    J_p = probability_current(model, x_s, t_rep).numpy().flatten()
    pm  = np.trapz(J_p, x_grid)      # this equals ⟨p⟩ for real wave-packet
    pmean_pinn.append(pm)

xmean_pinn = np.array(xmean_pinn)
pmean_pinn = np.array(pmean_pinn)
norm_time  = np.array(norm_time)

# Exact Ehrenfest values (V=0 free particle)
xmean_exact = x0 + k0 * t_fine
pmean_exact = np.full_like(t_fine, k0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#0d1117')
fig.suptitle('Ehrenfest Theorem Verification: Classical Trajectories from PINN Solution', 
             color='#e6edf3', fontsize=13)
for ax in axes:
    ax.set_facecolor('#161b22')
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
    ax.tick_params(colors='#8b949e')

axes[0].plot(t_fine, xmean_exact, '-',  color='#3fb950', lw=2.5, label='Exact $⟨x⟩ = x_0+k_0 t$')
axes[0].plot(t_fine, xmean_pinn,  '--', color='#58a6ff', lw=1.8, label='PINN $⟨x⟩$')
axes[0].set_xlabel('$t$', color='#8b949e'); axes[0].set_ylabel('$⟨x⟩(t)$', color='#8b949e')
axes[0].set_title('Position Expectation Value', color='#e6edf3')
axes[0].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')

axes[1].plot(t_fine, pmean_exact, '-',  color='#ffa657', lw=2.5, label='Exact $⟨p⟩ = k_0$')
axes[1].plot(t_fine, pmean_pinn,  '--', color='#d2a8ff', lw=1.8, label='PINN $⟨p⟩$')
axes[1].set_xlabel('$t$', color='#8b949e'); axes[1].set_ylabel('$⟨p⟩(t)$', color='#8b949e')
axes[1].set_title('Momentum Expectation Value', color='#e6edf3')
axes[1].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')

axes[2].plot(t_fine, norm_time, '-', color='#ff7b72', lw=2)
axes[2].axhline(1.0, color='#30363d', lw=1.2, ls='--', label='Target norm = 1')
axes[2].set_xlabel('$t$', color='#8b949e'); axes[2].set_ylabel('$\\int|\\hat\\psi|^2 dx$', color='#8b949e')
axes[2].set_title('Norm Conservation $\\int|\\psi|^2 dx$ vs $t$', color='#e6edf3')
axes[2].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')

plt.tight_layout()
plt.savefig('../outputs/schrodinger_ehrenfest.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

err_x = np.mean(np.abs(xmean_pinn - xmean_exact))
err_p = np.mean(np.abs(pmean_pinn - pmean_exact))
err_norm = np.std(norm_time)
print(f'Mean |⟨x⟩_pinn − ⟨x⟩_exact|  = {err_x:.5f}')
print(f'Mean |⟨p⟩_pinn − ⟨p⟩_exact|  = {err_p:.5f}')
print(f'Norm std dev ∫|ψ|²dx over t   = {err_norm:.5f}')
print('Saved → outputs/schrodinger_ehrenfest.png')

In [ ]:
# ── Phase Visualization: θ(x,t) = arctan(ψ_i / ψ_r) ─────────────────────
# The phase encodes momentum structure: ∂θ/∂x = local wave-number k(x,t)
# For the wavepacket: θ(x,0) = k0·x → uniform spatial frequency k0

t_phase_snaps = [0.0, 0.3, 0.6, 1.0]
fig, axes = plt.subplots(2, len(t_phase_snaps), figsize=(18, 8), facecolor='#0d1117')
fig.suptitle('Phase $\\theta(x,t)=\\arg\\psi$ and Local Wavenumber $\\partial\\theta/\\partial x$',
             color='#e6edf3', fontsize=13)

for j, tj in enumerate(t_phase_snaps):
    t_rep = torch.full((400, 1), tj)
    with torch.no_grad():
        pr, pi = model(x_s, t_rep)
    pr_n = pr.numpy().flatten()
    pi_n = pi.numpy().flatten()
    rho_n = pr_n**2 + pi_n**2

    # Phase (unwrapped for continuity)
    theta_pinn  = np.arctan2(pi_n, pr_n)
    theta_unwrp = np.unwrap(theta_pinn)

    # Exact phase
    pr_e, pi_e = exact_wavepacket(x_np, tj, x0, k0, sigma)
    theta_exact = np.unwrap(np.arctan2(pi_e, pr_e))

    # Local wavenumber  k(x,t) = ∂θ/∂x
    k_pinn  = np.gradient(theta_unwrp, x_np)
    k_exact = np.gradient(theta_exact, x_np)

    # Mask low-density regions (phase undefined there)
    mask = rho_n > 0.005 * rho_n.max()

    ax_top = axes[0, j]
    ax_top.set_facecolor('#161b22')
    for sp in ax_top.spines.values(): sp.set_edgecolor('#30363d')
    ax_top.plot(x_np[mask], theta_exact[mask], '-', color='#ffa657', lw=2.5, label='Exact $\\theta$')
    ax_top.plot(x_np[mask], theta_unwrp[mask], '--', color='#58a6ff', lw=1.8, label='PINN $\\theta$')
    ax_top.set_title(f'$t={tj:.1f}$', color='#e6edf3')
    ax_top.set_xlabel('$x$', color='#8b949e')
    ax_top.tick_params(colors='#8b949e')
    if j == 0:
        ax_top.set_ylabel('$\\theta(x,t)$', color='#8b949e')
        ax_top.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=8)

    ax_bot = axes[1, j]
    ax_bot.set_facecolor('#161b22')
    for sp in ax_bot.spines.values(): sp.set_edgecolor('#30363d')
    ax_bot.plot(x_np[mask], k_exact[mask], '-', color='#3fb950', lw=2.5, label='Exact $k(x,t)$')
    ax_bot.plot(x_np[mask], k_pinn[mask],  '--', color='#d2a8ff', lw=1.8, label='PINN $k(x,t)$')
    ax_bot.axhline(k0, color='#ff7b72', lw=0.8, ls=':', alpha=0.7, label=f'$k_0={k0}$')
    ax_bot.set_xlabel('$x$', color='#8b949e')
    ax_bot.tick_params(colors='#8b949e')
    if j == 0:
        ax_bot.set_ylabel('$k(x,t) = \\partial\\theta/\\partial x$', color='#8b949e')
        ax_bot.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/schrodinger_phase.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved → outputs/schrodinger_phase.png')

# ── Summary Metrics Table ────────────────────────────────────────────────
t_eval_pts = np.linspace(T_MIN, T_MAX, 20)
results_table = []

for tj in t_eval_pts:
    t_rep = torch.full((400, 1), tj)
    with torch.no_grad():
        pr, pi = model(x_s, t_rep)
    rho_p = (pr**2 + pi**2).numpy().flatten()
    pr_e, pi_e = exact_wavepacket(x_np, tj, x0, k0, sigma)
    rho_e = pr_e**2 + pi_e**2
    rel_rho = np.sqrt(np.trapz((rho_p - rho_e)**2, x_np) / np.trapz(rho_e**2, x_np))
    results_table.append({'t': tj, 'rel_l2_rho': rel_rho,
                          'norm_pinn': np.trapz(rho_p, x_np),
                          'norm_exact': np.trapz(rho_e, x_np)})

df_schrodinger = pd.DataFrame(results_table)
df_schrodinger.to_csv('../outputs/schrodinger_benchmark.csv', index=False)

print('\nSaved → outputs/schrodinger_benchmark.csv')
print(df_schrodinger[df_schrodinger.t.isin([0.0, 0.25, 0.5, 0.75, 1.0])].to_string(
      index=False, float_format='{:.5f}'.format))